# Study on the duration (risk) premium

In [1]:
import os
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

plt.style.use('seaborn-v0_8-whitegrid')
example_root = Path(os.path.abspath(""))

## Data loading

In [2]:
eurirs_df = pd.read_csv(example_root / "data" / f"EURIRS_history_2000-01-01_to_2025-12-31.csv")
eurbmk_df = pd.read_csv(example_root / "data" / f"0#EUBMK=_history_2000-01-01_to_2025-12-31.csv")

In [3]:
buckets = dict([(f"{i}M", i / 12.) for i in range(1, 19)] + [(f"{i}Y", i) for i in range(1, 51)])

In [4]:
# Rename the columns for EURIRS Curve
eurirs_map = {}

for col in eurirs_df.columns:
  if col == "Date":
    continue
  
  col_mat = col.split("=")[0][-3:]
  
  try:
    col_ttm  = int(col_mat[:2])
    col_mat = col_mat
  except Exception as e:
    col_mat = col_mat[1:]
  
  eurirs_map[col] = col_mat
  
eurirs_df.rename(columns=eurirs_map, inplace=True)

In [5]:
# Rename the columns for 0#EUBMK= Curve
eurbmk_map = {}

for col in eurbmk_df.columns:
  if col == "Date":
    continue
  
  col_mat = col.split("=")[0][-4:-1]
  
  try:
    col_ttm  = int(col_mat[:2])
    col_mat = col_mat
  except Exception as e:
    col_mat = col_mat[1:]
  
  eurbmk_map[col] = col_mat
  
eurbmk_df.rename(columns=eurbmk_map, inplace=True)

In [6]:
data_color  = "royalblue"
model_color = "indianred"

## Risk premiums for 1Y holding period

Consider the 1Y spot rate on a given yield curve. If a zero-coupon bond with said 1Y to maturity could be purchased, the return of holding it for a year is known in advance (the YTM). Fama (1984b) defines the premium as 

$$
  \Pi^{(t)}_\tau = H^{(t)}_\tau - H^{(t)}_1,
$$

where $H^{(t)}_i$ is the observed (continuously) compunding single period rate of return for maturity $i$ over the interval $t - 1$ to $t$.

The holding period return is defined as (assuming no coupons)

$$
  H^{(t)}_\tau = \frac{P^{(t)}_{\tau} - P^{(t-1)}_{\tau + 1}}{P^{(t-1)}_{\tau + 1}} ,
$$

where $P^{(t)}_i$ is the price of a zero-coupon bond with maturity $i$ at time $t$. 

Obviously, the return for bonds with longer than 1Y to maturity is dependent on the development of interest rates in the coming year. Thus, the risk premium needs to be studied statistically.

In [7]:
eurirs_df = eurirs_df[["Date", "1Y", "2Y", "3Y", "4Y", "5Y", "6Y", "7Y", "8Y", "9Y", "10Y", "29Y", "30Y"]]
eurbmk_df = eurbmk_df[["Date", "1Y", "2Y", "3Y", "4Y", "5Y", "6Y", "7Y", "8Y", "9Y", "10Y"]]
eurirs_df.index = pd.DatetimeIndex(eurirs_df["Date"])
eurbmk_df.index = pd.DatetimeIndex(eurbmk_df["Date"])
eurirs_df.drop(columns=["Date"], inplace=True)
eurbmk_df.drop(columns=["Date"], inplace=True)

In [8]:
# Find the spot rates shifted by 1Y
for i in range(1, 11):
  eurirs_df[f"S{i}Y1Y"] = eurirs_df[f"{i}Y"].shift(periods=365, freq='D').astype(np.float64)
  eurbmk_df[f"S{i}Y1Y"] = eurbmk_df[f"{i}Y"].shift(periods=365, freq='D').astype(np.float64)
  
eurirs_df[f"S30Y1Y"] = eurirs_df[f"30Y"].shift(periods=365, freq='D').astype(np.float64)

In [9]:
# Find the holding period returns
for i in range(2, 11):
  eurirs_df[f"H{i}Y{i-1}Y"] = (np.exp(-eurirs_df[f"{i - 1}Y"].to_numpy() / 100 * (i - 1)) - np.exp(-eurirs_df[f"S{i}Y1Y"].to_numpy() / 100 * i)) / np.exp(-eurirs_df[f"S{i}Y1Y"].to_numpy() / 100 * i)
  eurbmk_df[f"H{i}Y{i-1}Y"] = (np.exp(-eurbmk_df[f"{i - 1}Y"].to_numpy() / 100 * (i - 1)) - np.exp(-eurbmk_df[f"S{i}Y1Y"].to_numpy() / 100 * i)) / np.exp(-eurbmk_df[f"S{i}Y1Y"].to_numpy() / 100 * i)

eurirs_df[f"H1Y0Y"] = (1. - np.exp(-eurirs_df[f"S1Y1Y"].to_numpy() / 100 * 1.)) / np.exp(-eurirs_df[f"S1Y1Y"].to_numpy() / 100 * 1.)
eurbmk_df[f"H1Y0Y"] = (1. - np.exp(-eurbmk_df[f"S1Y1Y"].to_numpy() / 100 * 1.)) / np.exp(-eurbmk_df[f"S1Y1Y"].to_numpy() / 100 * 1.)

eurirs_df[f"H30Y29Y"] = (np.exp(-eurirs_df[f"29Y"].to_numpy() / 100 * (i - 1)) - np.exp(-eurirs_df[f"S30Y1Y"].to_numpy() / 100 * i)) / np.exp(-eurirs_df[f"S30Y1Y"].to_numpy() / 100 * i)

In [10]:
eurirs_df = eurirs_df.dropna()
eurbmk_df = eurbmk_df.dropna()

In [11]:
eurirs_df.head()

,1Y,2Y,3Y,4Y,5Y,6Y,7Y,8Y,9Y,10Y,...,H3Y2Y,H4Y3Y,H5Y4Y,H6Y5Y,H7Y6Y,H8Y7Y,H9Y8Y,H10Y9Y,H1Y0Y,H30Y29Y
Date,,,,,,,,,,,,,,,,,,,,,
2001-07-11,4.375,4.4700,4.6125,4.7700,4.920,5.0675,5.2000,5.3150,5.4075,5.480,...,0.076430,0.085157,0.091661,0.096474,0.099219,0.100979,0.101144,0.100842,0.051429,0.073286
2001-07-12,4.365,4.4750,4.6150,4.7310,4.886,5.0300,5.1790,5.2920,5.3830,5.455,...,0.077388,0.087031,0.096239,0.101640,0.106138,0.107240,0.108647,0.109634,0.052797,0.082367
2001-07-13,4.395,4.4850,4.6425,4.7850,4.936,5.0825,5.2100,5.3225,5.4150,5.484,...,0.077981,0.086352,0.093737,0.098725,0.101502,0.103514,0.103708,0.104232,0.053007,0.077130
2001-07-17,4.365,4.4175,4.5675,4.7275,4.870,4.9925,5.1175,5.2250,5.3125,5.380,...,0.081772,0.091961,0.099769,0.107096,0.112712,0.116027,0.118323,0.120500,0.053534,0.088010
2001-07-18,4.335,4.4500,4.5800,4.6990,4.851,4.9930,5.1260,5.2340,5.3220,5.396,...,0.080420,0.091551,0.101078,0.107217,0.111900,0.114471,0.116814,0.117417,0.053639,0.088173


In [12]:
eurbmk_df.head()

,1Y,2Y,3Y,4Y,5Y,6Y,7Y,8Y,9Y,10Y,...,H2Y1Y,H3Y2Y,H4Y3Y,H5Y4Y,H6Y5Y,H7Y6Y,H8Y7Y,H9Y8Y,H10Y9Y,H1Y0Y
Date,,,,,,,,,,,,,,,,,,,,,
2003-06-10,1.818,1.8805,2.2140,2.4975,2.567,2.853,3.067,3.250,3.4630,3.5250,...,0.066690,0.095795,0.122322,0.148521,0.177402,0.194760,0.211271,0.218865,0.220097,0.037486
2003-06-11,1.822,1.8920,2.2110,2.4855,2.572,2.848,3.054,3.233,3.4565,3.5200,...,0.066647,0.095346,0.122019,0.149762,0.178238,0.196122,0.214169,0.220084,0.220566,0.037486
2003-06-12,1.830,1.9085,2.2335,2.4890,2.580,2.853,3.061,3.232,3.3955,3.5155,...,0.065901,0.093835,0.119873,0.146960,0.174380,0.192504,0.209842,0.216126,0.222998,0.037174
2003-06-13,1.818,1.8725,2.1940,2.4590,2.550,2.818,3.021,3.192,3.4005,3.4620,...,0.064856,0.093179,0.118916,0.145298,0.174168,0.189710,0.207570,0.214491,0.216837,0.037071
2003-06-17,1.876,1.9875,2.3025,2.6220,2.741,3.004,3.197,3.353,3.5370,3.5910,...,0.063707,0.089752,0.114053,0.136093,0.158690,0.175067,0.191353,0.197169,0.200238,0.036863


### Mean risk premium

#### Full dataset

In [13]:
for i in range(2, 11):
  eurirs_mean_risk_prem = np.mean(eurirs_df[f"H{i}Y{i-1}Y"].to_numpy() - eurirs_df[f"H1Y0Y"].to_numpy())
  eurbmk_mean_risk_prem = np.mean(eurbmk_df[f"H{i}Y{i-1}Y"].to_numpy() - eurbmk_df[f"H1Y0Y"].to_numpy())
  
  print(f"Risk premium for maturity {i}Y:")
  print(f"EURIRS: {eurirs_mean_risk_prem:.3f}")
  print(f"EURBMK: {eurbmk_mean_risk_prem:.3f}")
  
eurirs_mean_risk_prem = np.mean(eurirs_df[f"H30Y29Y"].to_numpy() - eurirs_df[f"H1Y0Y"].to_numpy())
print(f"Risk premium for maturity 30Y:")
print(f"EURIRS: {eurirs_mean_risk_prem:.3f}")

Risk premium for maturity 2Y:
EURIRS: 0.003
EURBMK: 0.001
Risk premium for maturity 3Y:
EURIRS: 0.007
EURBMK: 0.004
Risk premium for maturity 4Y:
EURIRS: 0.011
EURBMK: 0.009
Risk premium for maturity 5Y:
EURIRS: 0.015
EURBMK: 0.012
Risk premium for maturity 6Y:
EURIRS: 0.018
EURBMK: 0.016
Risk premium for maturity 7Y:
EURIRS: 0.022
EURBMK: 0.019
Risk premium for maturity 8Y:
EURIRS: 0.025
EURBMK: 0.022
Risk premium for maturity 9Y:
EURIRS: 0.028
EURBMK: 0.025
Risk premium for maturity 10Y:
EURIRS: 0.030
EURBMK: 0.027
Risk premium for maturity 30Y:
EURIRS: 0.026


#### Financial crisis (2007-2009)

In [14]:
tmp_eurirs_df = eurirs_df[(eurirs_df.index >= "2007-01-01") & (eurirs_df.index <= "2009-12-31")]
tmp_eurbmk_df = eurbmk_df[(eurbmk_df.index >= "2007-01-01") & (eurbmk_df.index <= "2009-12-31")]

for i in range(2, 11):
  eurirs_mean_risk_prem = np.mean(tmp_eurirs_df[f"H{i}Y{i-1}Y"].to_numpy() - tmp_eurirs_df[f"H1Y0Y"].to_numpy())
  eurbmk_mean_risk_prem = np.mean(tmp_eurbmk_df[f"H{i}Y{i-1}Y"].to_numpy() - tmp_eurbmk_df[f"H1Y0Y"].to_numpy())
  
  print(f"Risk premium for maturity {i}Y:")
  print(f"EURIRS: {eurirs_mean_risk_prem:.3f}")
  print(f"EURBMK: {eurbmk_mean_risk_prem:.3f}")
  
eurirs_mean_risk_prem = np.mean(tmp_eurirs_df[f"H30Y29Y"].to_numpy() - tmp_eurirs_df[f"H1Y0Y"].to_numpy())
print(f"Risk premium for maturity 30Y:")
print(f"EURIRS: {eurirs_mean_risk_prem:.3f}")

Risk premium for maturity 2Y:
EURIRS: 0.006
EURBMK: 0.008
Risk premium for maturity 3Y:
EURIRS: 0.013
EURBMK: 0.017
Risk premium for maturity 4Y:
EURIRS: 0.016
EURBMK: 0.023
Risk premium for maturity 5Y:
EURIRS: 0.018
EURBMK: 0.021
Risk premium for maturity 6Y:
EURIRS: 0.020
EURBMK: 0.029
Risk premium for maturity 7Y:
EURIRS: 0.021
EURBMK: 0.023
Risk premium for maturity 8Y:
EURIRS: 0.022
EURBMK: 0.026
Risk premium for maturity 9Y:
EURIRS: 0.023
EURBMK: 0.023
Risk premium for maturity 10Y:
EURIRS: 0.023
EURBMK: 0.024
Risk premium for maturity 30Y:
EURIRS: 0.015


#### Euro area crisis (2010-2014)

In [15]:
tmp_eurirs_df = eurirs_df[(eurirs_df.index >= "2010-01-01") & (eurirs_df.index <= "2014-12-31")]
tmp_eurbmk_df = eurbmk_df[(eurbmk_df.index >= "2010-01-01") & (eurbmk_df.index <= "2014-12-31")]

for i in range(2, 11):
  eurirs_mean_risk_prem = np.mean(tmp_eurirs_df[f"H{i}Y{i-1}Y"].to_numpy() - tmp_eurirs_df[f"H1Y0Y"].to_numpy())
  eurbmk_mean_risk_prem = np.mean(tmp_eurbmk_df[f"H{i}Y{i-1}Y"].to_numpy() - tmp_eurbmk_df[f"H1Y0Y"].to_numpy())
  
  print(f"Risk premium for maturity {i}Y:")
  print(f"EURIRS: {eurirs_mean_risk_prem:.3f}")
  print(f"EURBMK: {eurbmk_mean_risk_prem:.3f}")
  
eurirs_mean_risk_prem = np.mean(tmp_eurirs_df[f"H30Y29Y"].to_numpy() - tmp_eurirs_df[f"H1Y0Y"].to_numpy())
print(f"Risk premium for maturity 30Y:")
print(f"EURIRS: {eurirs_mean_risk_prem:.3f}")

Risk premium for maturity 2Y:
EURIRS: 0.006
EURBMK: 0.006
Risk premium for maturity 3Y:
EURIRS: 0.016
EURBMK: 0.014
Risk premium for maturity 4Y:
EURIRS: 0.025
EURBMK: 0.027
Risk premium for maturity 5Y:
EURIRS: 0.035
EURBMK: 0.037
Risk premium for maturity 6Y:
EURIRS: 0.043
EURBMK: 0.048
Risk premium for maturity 7Y:
EURIRS: 0.050
EURBMK: 0.056
Risk premium for maturity 8Y:
EURIRS: 0.057
EURBMK: 0.064
Risk premium for maturity 9Y:
EURIRS: 0.063
EURBMK: 0.069
Risk premium for maturity 10Y:
EURIRS: 0.068
EURBMK: 0.073
Risk premium for maturity 30Y:
EURIRS: 0.053


## Hotelling $t^2$ test